# Quads design (3dp pla + blue shims) for multiloading source cloaking


## Imports

In [ ]:
import os

os.environ["XLA_FLAGS"] = (
    "--xla_force_host_platform_device_count=8"  # Use 8 CPU cores for JAX pmap
)

import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from pathlib import Path
from problems.quads_dynamic_load_shielding_source_multi_loading import (
    ForwardInput,
    ForwardProblem,
    OptimizationProblem,
)
from mechanicalmetamaterialcloaks.utils import save_data, load_data, SolutionData
from mechanicalmetamaterialcloaks.plotting import plot_geometry, plot_geometry_field_overlaid

from typing import Optional, List

jax.config.update("jax_enable_x64", True)  # enable float64 type

plt.style.use(["science", "grid"])
%matplotlib widget

## Plotting functions


In [ ]:
def plot_objective_iterations(
    optimization: OptimizationProblem,
    optimization_filename: Optional[str] = None,
    figsize=(10, 9),
):
    fig, axes = plt.subplots(figsize=figsize, sharex=True, constrained_layout=True)
    axes.set(ylabel="Objective")
    axes.plot(
        jnp.array(optimization.objective_values)
        / optimization.forward_problem.n_timepoints,
        lw=3,
        color="#16a085",
        label="Total objective",
        zorder=10,
    )
    colors = ["#d35400", "#e67e22", "#f39c12", "#f1c40f"]

    for k in range(len(optimization.objective_values_individual[0])):
        axes.plot(
            [
                optimization.objective_values_individual[i][k]
                / optimization.forward_problem.n_timepoints
                for i in range(len(optimization.objective_values_individual))
            ],
            lw=2,
            color=colors[k],
            label=f"Loading direction {k + 1}",
            linestyle="--",
        )
    axes.axhline(y=0, color="black")
    axes.set(xlabel=r"Iteration \#")
    axes.legend()

    if optimization_filename is not None:
        path = Path(
            f"../out/{optimization.name}/{optimization_filename}/objective_iterations.png"
        )
        # Make sure parents directories exist
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(path), dpi=300)
    return fig, axes


def plot_objective_and_constraints_iterations(
    optimization: OptimizationProblem,
    optimization_filename: Optional[str] = None,
    figsize=(10, 9),
):
    fig, axes = plt.subplots(
        nrows=3, figsize=figsize, sharex=True, constrained_layout=True
    )
    axes[0].set(ylabel="Objective")
    axes[0].plot(
        optimization.objective_values, lw=3, color="#2980b9", label="Total objective"
    )
    colors = ["#d35400", "#e67e22", "#f39c12", "#f1c40f"]

    for k in range(len(optimization.objective_values_individual[0])):
        axes[0].plot(
            [
                optimization.objective_values_individual[i][k]
                for i in range(len(optimization.objective_values_individual))
            ],
            lw=3,
            color=colors[k],
            label=f"Loading direction {k + 1}",
            linestyle="--",
        )
    axes[0].legend()
    axes[1].set(ylabel="Angle constraints violation")
    axes[1].plot(optimization.constraints_violation["angles"], lw=3, color="#c0392b")
    axes[1].axhline(y=0, color="black")
    axes[2].set(ylabel="Edge length constraints violation")
    axes[2].plot(
        optimization.constraints_violation["edge_lengths"], lw=3, color="#c0392b"
    )
    axes[2].axhline(y=0, color="black")
    axes[-1].set(xlabel=r"Iteration \#")

    if optimization_filename is not None:
        path = Path(
            f"../out/{optimization.name}/{optimization_filename}/objective_and_constraints_iterations.png"
        )
        # Make sure parents directories exist
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(path), dpi=300)
    return fig, axes


def compute_individual_objective(
    solution_data: SolutionData, optimization: OptimizationProblem
):
    dimension_less = jnp.array(
        [
            1 / optimization.forward_problem.cloaked_geometry.spacing,
            1 / optimization.forward_problem.cloaked_geometry.spacing,
            1,
        ]
    )
    return (
        (
            solution_data.fields[
                :, 0, optimization.forward_problem.mgIDs_surronding_area, :
            ]
            * dimension_less
        )
        ** 2
    ).sum()


def delta_function_integrated(solution_data, spacing, mgIDs_surronding_area):
    dimension_less = jnp.array([1 / spacing, 1 / spacing, 1])
    return (
        (
            ((solution_data.fields[:, 0, mgIDs_surronding_area, :]) * dimension_less)
            ** 2
        ).sum(axis=(1, 2))
    ) ** 0.5


def generate_delta_difference_column(
    solution_data: List[SolutionData],
    mgIDs_surronding_area: jnp.ndarray,
    spacing: float,
    title: str,
    out_filename,
    solution_data_opcg: List[SolutionData] = None,
    sub_title: List = None,
    figsize=(12, 12),
):
    # plot
    fig, ax = plt.subplots(len(solution_data), figsize=figsize, constrained_layout=True)
    label = "initial guess" if solution_data_opcg != None else ""
    for k, sol_data in enumerate(solution_data):
        delta_value = delta_function_integrated(
            sol_data, spacing, mgIDs_surronding_area
        )
        timepoints = sol_data.timepoints
        if sub_title != None:
            ax[k].set_title(title + " - " + sub_title[k])
        ax[k].plot(timepoints, delta_value, label=label)

        ax[k].set_ylabel("normalized delta")
    ax[len(solution_data) - 1].set_xlabel("Time")
    if solution_data_opcg != None:
        for k, sol_data_opcg in enumerate(solution_data_opcg):
            delta_value_opcg = delta_function_integrated(
                sol_data_opcg, spacing, mgIDs_surronding_area
            )
            ax[k].plot(timepoints, delta_value_opcg, label="optimized", linestyle="--")
            ax[k].legend(fancybox=True, framealpha=0.9)
    fig.savefig(out_filename + ".png", bbox_inches="tight", dpi=300)
    return fig, ax


def generate_delta_difference(
    solution_data: List[SolutionData],
    mgIDs_surronding_area: jnp.ndarray,
    spacing: float,
    title: str,
    out_filename,
    solution_data_opcg: List[SolutionData] = None,
    sub_title: List = None,
    figsize=(12, 12),
):
    # plot
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    label = "initial guess" if solution_data_opcg != None else ""
    for k, sol_data in enumerate(solution_data):
        delta_value = delta_function_integrated(
            sol_data, spacing, mgIDs_surronding_area
        )
        timepoints = sol_data.timepoints
        ax.plot(timepoints, delta_value, label=label + " - " + sub_title[k])

    if solution_data_opcg != None:
        for k, sol_data_opcg in enumerate(solution_data_opcg):
            delta_value_opcg = delta_function_integrated(
                sol_data_opcg, spacing, mgIDs_surronding_area
            )
            ax.plot(
                timepoints,
                delta_value_opcg,
                label="optimized" + " - " + sub_title[k],
                linestyle="--",
            )
            ax.legend(fancybox=True, framealpha=0.9)

    ax.set_title(title)
    ax.set_ylabel("normalized delta")
    ax.set_xlabel("Time")
    fig.savefig(out_filename + ".png", bbox_inches="tight", dpi=300)
    return fig, ax

## Problem setup


In [ ]:
# NOTE: Units are mm, N, s

# Geometrical params
n1_blocks = 31
n2_blocks = 31
spacing = 15.0  # mm
hinge_length = 0.15 * spacing
initial_angle = 20 * jnp.pi / 180

# choose the limits of the excited blocks
x0, y0 = n1_blocks // 2 * spacing, n2_blocks // 2 * spacing
r0 = spacing / 2
N = 25
excited_blocks_limits = [
    [r0 * jnp.cos(2 * jnp.pi * k / N) + x0, r0 * jnp.sin(2 * jnp.pi * k / N) + y0]
    for k in range(N + 1)
]

# choose the limits of the shielding cloak blocks
cloak_size = 8.5
r1 = cloak_size * spacing  # 3.8
N = 40
shielding_cloak_limits = [
    [r1 * jnp.cos(2 * jnp.pi * k / N) + x0, r1 * jnp.sin(2 * jnp.pi * k / N) + y0]
    for k in range(N + 1)
]

# Mechanical params
k_stretch = 120.0  # stretching stiffness 120. N/mm
k_shear = 1.19  # shearing stiffness 1.19 N/mm
k_rot = 1.50  # rotational stiffness 1.50 Nmm
density = 6.18e-9  # Mg/mm^2

damping_scaling = 1.0
damping = (
    0.0186
    * jnp.array(
        [
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.36125 * density * spacing**2 * k_shear) ** 0.5,
            2 * (0.02175026 * density * spacing**4 * k_rot) ** 0.5,
        ]
    )
    * damping_scaling
)

# boundary conditions
clamping_corners = True

# Forward input
amplitude = 0.3 * spacing
loading_rate = 2.0  # Hz
loading_angle = 0 * jnp.pi / 180
n_timepoints = 200
simulation_time = 2.0 / loading_rate  # s
optimization_name = "quads_shielding_multi_loading_source_3dp_pla_shims_4_loads"


# Forward problem
problem = ForwardProblem(
    n1_blocks=n1_blocks,
    n2_blocks=n2_blocks,
    spacing=spacing,
    bond_length=hinge_length,
    shielding_cloak_limits=shielding_cloak_limits,
    excited_blocks_limits=excited_blocks_limits,
    horizontal_vertical_shifts_mg=None,
    initial_angle=initial_angle,
    k_stretch=k_stretch,
    k_shear=k_shear,
    k_rot=k_rot,
    density=density,
    k_contact=k_rot,
    damping=damping,
    min_angle=-15 * jnp.pi / 180,
    cutoff_angle=-10 * jnp.pi / 180,
    clamping_corners=clamping_corners,
    simulation_time=simulation_time,
    n_timepoints=n_timepoints,
    name=optimization_name,
)

problem.setup()
problem.plot_sketch()

In [ ]:
design_value = problem.all_to_cloak_shifts(problem.horizontal_vertical_shifts_mg)

forward_input = ForwardInput(
    horizontal_shifts=design_value[0],
    vertical_shifts=design_value[1],
    amplitude=(amplitude, amplitude, amplitude, amplitude),
    loading_rate=(loading_rate, loading_rate, loading_rate, loading_rate),
    loading_angle=(
        0 * jnp.pi / 180,
        45 * jnp.pi / 180,
        90 * jnp.pi / 180,
        135 * jnp.pi / 180,
    ),
)

# Optimization
weights = (0.25, 0.25, 0.25, 0.25)
optimization = OptimizationProblem(
    forward_problem=problem,
    forward_input=forward_input,
    name=optimization_name,
    weights=weights,
)

problem_filename_prefix = f"quads_{optimization.forward_problem.n1_blocks}x{optimization.forward_problem.n2_blocks}_amplitude_{amplitude:.2f}_loading_rate_{loading_rate}_initial_angle_{initial_angle * 180 / jnp.pi:.1f}_clamping_corners_{clamping_corners}_damping_scaling_{damping_scaling}"
min_edge_length = 3
min_block_angle = 30
optimization_filename = f"opt_integrated_with_angle_{min_block_angle}_and_length_{min_edge_length}_constraints_{problem_filename_prefix}_cloak_size_{cloak_size:.1f}"

## Optimization

### Import most recent optimization object


In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl"
    )
)

### Run optimization

In [ ]:
# optimization.run_optimization_nlopt(
#     initial_guess=(
#         optimization.forward_input.horizontal_shifts,
#         optimization.forward_input.vertical_shifts,
#     ),
#     # initial_guess=optimization.design_values[-1],
#     n_iterations=50,
#     min_block_angle=min_block_angle * jnp.pi / 180,
#     min_void_angle=0 * jnp.pi / 180,
#     min_edge_length=min_edge_length * 1.0,  # mm
#     max_time=12 * 60 * 60,  # 12 hours
# )

# save_data(
#     f"../data/{optimization.name}/{optimization_filename}.pkl",
#     optimization.to_dict(),  # Optimization problem
# )


## Plots

### Import most recent optimization object


In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl",
    )
)

### Objective iterations


In [ ]:
fig, axes = plot_objective_and_constraints_iterations(
    optimization=optimization,
    figsize=(6, 6),
    optimization_filename=optimization_filename,
)
# axes.grid(False)
# # Spines 1pt thick
# for spine in axes.spines.values():
#     spine.set_linewidth(1)
# axes.tick_params(labelsize=16)
# axes.set_xlabel(r"Iteration \#", fontsize=18)
# axes.set_ylabel("Objective", fontsize=18)
# # Remove legend
# axes.get_legend().remove()

### Plot designs


In [ ]:
plt.ioff()
for solution_data, label in zip(
    [
        optimization.forward_problem.solution_data["mg"],
        optimization.forward_problem.solution_data["opcg"][0],
    ],
    ["reference", "cloaked"],
):
    fig, axes = plot_geometry(
        block_centroids=solution_data.block_centroids,
        centroid_node_vectors=solution_data.centroid_node_vectors,
        bond_connectivity=solution_data.bond_connectivity,
        figsize=(4, 4),
    )
    axes.axis("off")
    fig.savefig(
        f"../out/{optimization.name}/{optimization_filename}/{label}_geometry.png",
        dpi=300,
        transparent=True,
    )
    plt.close(fig)
plt.ion()

## Compare different cloak sizes

In [ ]:
optimization_filenames = [
    "opt_integrated_with_angle_30_and_length_3_constraints_quads_31x31_amplitude_4.50_loading_rate_2.0_initial_angle_20.0_clamping_corners_True_damping_scaling_1.0_cloak_size_2.5",
    "opt_integrated_with_angle_30_and_length_3_constraints_quads_31x31_amplitude_4.50_loading_rate_2.0_initial_angle_20.0_clamping_corners_True_damping_scaling_1.0_cloak_size_4.5",
    "opt_integrated_with_angle_30_and_length_3_constraints_quads_31x31_amplitude_4.50_loading_rate_2.0_initial_angle_20.0_clamping_corners_True_damping_scaling_1.0_cloak_size_6.5",
    "opt_integrated_with_angle_30_and_length_3_constraints_quads_31x31_amplitude_4.50_loading_rate_2.0_initial_angle_20.0_clamping_corners_True_damping_scaling_1.0_cloak_size_8.5",
]
optimizations = [
    OptimizationProblem.from_dict(
        load_data(f"../data/{optimization.name}/{optimization_filename}.pkl")
    )
    for optimization_filename in optimization_filenames
]
for optimization in optimizations:
    optimization.forward_problem.setup()

### Plot designs


In [ ]:
plt.ioff()
for optimization, optimization_filename in zip(optimizations, optimization_filenames):
    for solution_data, label in zip(
        [
            optimization.forward_problem.solution_data["mg"],
            optimization.forward_problem.solution_data["opcg"][0],
        ],
        ["reference", "cloaked"],
    ):
        fig, axes = plot_geometry(
            block_centroids=solution_data.block_centroids,
            centroid_node_vectors=solution_data.centroid_node_vectors,
            bond_connectivity=solution_data.bond_connectivity,
            figsize=(4, 4),
        )
        axes.axis("off")
        out_path = Path(f"../out/{optimization.name}/{optimization_filename}/{label}_geometry.png")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            out_path,
            dpi=300,
            transparent=True,
        )
        plt.close(fig)
plt.ion()

In [ ]:
plt.ioff()
for optimization, optimization_filename in zip(optimizations, optimization_filenames):
    for solution_data, label in zip(
        [
            optimization.forward_problem.solution_data["ig"],
            optimization.forward_problem.solution_data["opcg"],
        ],
        ["reference", "cloaked"],
    ):
        peak_normalized_displacement = jnp.mean(
            jnp.array(
                [
                    jnp.linalg.norm(
                        data.fields[:, 0, :, :]
                        * jnp.array(
                            [
                                1 / optimization.forward_problem.spacing,
                                1 / optimization.forward_problem.spacing,
                                1.0,
                            ]
                        ),
                        axis=-1,
                    ).max(axis=0)  # Max over timepoints
                    for data in solution_data
                ]
            ),
            axis=0,  # Average over loading directions
        )
        # Plot design colored by displacement field
        fig, axes = plot_geometry_field_overlaid(
            data=solution_data[0],
            timepoint=0,
            field="normalized_displacement",
            field_values=peak_normalized_displacement[None],
            cmap="inferno",
            vlim=(0, 0.3),
            figsize=(4, 4),
            colorbar=False,
        )
        axes.axis("off")
        out_path = Path(f"../out/{optimization.name}/{optimization_filename}/{label}_geometry_normalized_displacement.png")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            out_path,
            dpi=300,
            transparent=True,
        )
        plt.close(fig)
plt.ion()

In [ ]:
objective_values_cloak_sizes = jnp.array(
    [
        optimization.objective_values[-1]
        / optimization.forward_problem.n_timepoints
        / optimization.forward_problem.mgIDs_surronding_area.shape[0]
        for optimization in optimizations
    ]
)
objective_values_cloak_sizes_initial = jnp.array(
    [
        optimization.objective_values[0]
        / optimization.forward_problem.n_timepoints
        / optimization.forward_problem.mgIDs_surronding_area.shape[0]
        for optimization in optimizations
    ]
)
# Peak normalized displacement in the surrounding area
peak_cloaked_displacement_cloak_sizes = jnp.array(
    [
        jnp.mean(
            jnp.array(
                [
                    jnp.linalg.norm(
                        data.fields[
                            :, 0, optimization.forward_problem.mgIDs_surronding_area, :
                        ]
                        * jnp.array(
                            [
                                1 / optimization.forward_problem.spacing,
                                1 / optimization.forward_problem.spacing,
                                1.0,
                            ]
                        ),
                        axis=-1,
                    ).max()
                    for data in optimization.forward_problem.solution_data["opcg"]
                ]
            )
        )
        for optimization in optimizations
    ]
)
peak_surrounding_displacement_cloak_sizes = jnp.array(
    [
        jnp.mean(
            jnp.array(
                [
                    jnp.linalg.norm(
                        data.fields[
                            :, 0, optimization.forward_problem.mgIDs_surronding_area, :
                        ]
                        * jnp.array(
                            [
                                1 / optimization.forward_problem.spacing,
                                1 / optimization.forward_problem.spacing,
                                1.0,
                            ]
                        ),
                        axis=-1,
                    ).max()
                    for data in optimization.forward_problem.solution_data["ig"]
                ]
            )
        )
        for optimization in optimizations
    ]
)

# Peak normalized displacement on the boundary
boundary_block_ids = jnp.array(
    [
        *jnp.arange(optimization.forward_problem.n1_blocks),  # bottom row
        *jnp.arange(
            optimization.forward_problem.n1_blocks
            * (optimization.forward_problem.n2_blocks - 1),
            optimization.forward_problem.n1_blocks
            * optimization.forward_problem.n2_blocks,
        ),  # top row
        *jnp.arange(
            optimization.forward_problem.n1_blocks,
            optimization.forward_problem.n1_blocks
            * (optimization.forward_problem.n2_blocks - 1),
            optimization.forward_problem.n1_blocks,
        ),  # left column
        *jnp.arange(
            optimization.forward_problem.n1_blocks - 1,
            optimization.forward_problem.n1_blocks
            * (optimization.forward_problem.n2_blocks - 1)
            + optimization.forward_problem.n1_blocks
            - 1,
            optimization.forward_problem.n1_blocks,
        ),  # right column
    ]
)
peak_cloaked_displacement_boundary_cloak_sizes = jnp.array(
    [
        jnp.mean(
            jnp.array(
                [
                    jnp.linalg.norm(
                        data.fields[:, 0, boundary_block_ids, :]
                        * jnp.array(
                            [
                                1 / optimization.forward_problem.spacing,
                                1 / optimization.forward_problem.spacing,
                                1.0,
                            ]
                        ),
                        axis=-1,
                    ).max()
                    for data in optimization.forward_problem.solution_data["opcg"]
                ]
            )
        )
        for optimization in optimizations
    ]
)
peak_surrounding_displacement_boundary_cloak_sizes = jnp.array(
    [
        jnp.mean(
            jnp.array(
                [
                    jnp.linalg.norm(
                        data.fields[:, 0, boundary_block_ids, :]
                        * jnp.array(
                            [
                                1 / optimization.forward_problem.spacing,
                                1 / optimization.forward_problem.spacing,
                                1.0,
                            ]
                        ),
                        axis=-1,
                    ).max()
                    for data in optimization.forward_problem.solution_data["ig"]
                ]
            )
        )
        for optimization in optimizations
    ]
)

In [ ]:
# Make a bar plot for the peak cloaked displacement of different cloak sizes

plt.close("all")
fig, axes = plt.subplots(figsize=(5.75, 3.0), constrained_layout=True)
cloak_sizes_rounded = jnp.ceil(jnp.array([2.5, 4.5, 6.5, 8.5]))
axes.bar(
    x=jnp.arange(len(cloak_sizes_rounded)),
    height=peak_cloaked_displacement_boundary_cloak_sizes,
    width=0.6,
    color="#9b59b6",
)
axes.axhline(
    y=peak_surrounding_displacement_boundary_cloak_sizes.mean(),  # They are all the same so we can take the mean
    color="#ff9900",
    ls="--",
    lw=2,
)
axes.set_xlim(-0.75, 3.75)
axes.set_ylim(0, 0.35)
axes.set_xticks(jnp.arange(len(cloak_sizes_rounded)))
axes.set_xticklabels([f"{cloak_size:.0f}" for cloak_size in cloak_sizes_rounded])
axes.set_xlabel(r"Cloak radius [\# units]", fontsize=16)
axes.set_ylabel(
    r"Max boundary\\displacement $U^\text{max}_{\partial\Omega}$ [-]", fontsize=16
)
axes.tick_params(labelsize=14)
axes.grid(False)
# Spines 1pt thick
for spine in axes.spines.values():
    spine.set_linewidth(1)

out_path = Path(f"../out/{optimization.name}/{optimization_filenames[-1]}/peak_boundary_u_vs_cloak_size.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
# Make a bar plot for the peak cloaked displacement of different cloak sizes

plt.close("all")
fig, axes = plt.subplots(figsize=(5.75, 3.0), constrained_layout=True)
cloak_sizes_rounded = jnp.ceil(jnp.array([2.5, 4.5, 6.5, 8.5]))
axes.bar(
    x=jnp.arange(len(cloak_sizes_rounded)),
    height=peak_cloaked_displacement_cloak_sizes,
    width=0.6,
    color="#16a085",
)
axes.set_xlim(-0.75, 3.75)
axes.set_ylim(0, 0.2)
axes.set_xticks(jnp.arange(len(cloak_sizes_rounded)))
axes.set_xticklabels([f"{cloak_size:.0f}" for cloak_size in cloak_sizes_rounded])
axes.set_xlabel(r"Cloak radius [\# units]", fontsize=16)
axes.set_ylabel(r"Max normalized\\displacement $U_\text{max}$ [-]", fontsize=16)
axes.tick_params(labelsize=14)
axes.grid(False)
# Spines 1pt thick
for spine in axes.spines.values():
    spine.set_linewidth(1)
out_path = Path(f"../out/{optimization.name}/{optimization_filenames[-1]}/peak_u_vs_cloak_size.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=300, bbox_inches='tight')

In [ ]:
# max U_cloaked / max U_altered in the surrounding area
# Make a bar plot for the peak cloaked displacement of different cloak sizes

plt.close("all")
fig, axes = plt.subplots(figsize=(5.75, 3.0), constrained_layout=True)
cloak_sizes_rounded = jnp.ceil(jnp.array([2.5, 4.5, 6.5, 8.5]))
axes.bar(
    x=jnp.arange(len(cloak_sizes_rounded)),
    height=peak_cloaked_displacement_cloak_sizes
    / peak_surrounding_displacement_cloak_sizes,  # Factor out 1e-1
    width=0.6,
    color="#16a085",
)
axes.set_xlim(-0.75, 3.75)
axes.set_ylim(0, 1)
axes.set_xticks(jnp.arange(len(cloak_sizes_rounded)))
axes.set_xticklabels([f"{cloak_size:.0f}" for cloak_size in cloak_sizes_rounded])
axes.set_xlabel(r"Cloak radius [\# units]", fontsize=16)
axes.set_ylabel(
    r"$U_\text{max}^\text{cloaked}/U_\text{max}^\text{altered}$ [-]", fontsize=16
)
axes.tick_params(labelsize=14)
axes.grid(False)
# Spines 1pt thick
for spine in axes.spines.values():
    spine.set_linewidth(1)
out_path = Path(f"../out/{optimization.name}/{optimization_filenames[-1]}/peak_u_ratio_vs_cloak_size.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=300, bbox_inches='tight')

In [ ]:
# Make a bar plot for the objective values of different cloak sizes
plt.close("all")

for i in range(0, len(optimizations)):
    fig, axes = plt.subplots(figsize=(5, 4.1), constrained_layout=True)
    cloak_sizes_rounded = jnp.ceil(jnp.array([2.5, 4.5, 6.5, 8.5]))
    axes.bar(
        x=jnp.arange(len(cloak_sizes_rounded))[: i + 1],
        height=(
            peak_cloaked_displacement_cloak_sizes
            / peak_surrounding_displacement_cloak_sizes
        )[: i + 1],  # Factor out 1e-1
        width=0.6,
        color="#16a085",
    )
    axes.set_xlim(-0.75, 3.75)
    axes.set_ylim(0, 1)
    axes.set_xticks(jnp.arange(len(cloak_sizes_rounded)))
    axes.set_xticklabels([f"{cloak_size:.0f}" for cloak_size in cloak_sizes_rounded])
    axes.set_xlabel(r"Cloak radius [\# units]", fontsize=18)
    axes.set_ylabel("Normalized peak displacement [-]", fontsize=18)
    axes.tick_params(labelsize=16)
    axes.grid(False)
    # Spines 1pt thick
    for spine in axes.spines.values():
        spine.set_linewidth(1)
    out_path = Path(f"../out/{optimization.name}/{optimization_filenames[-1]}/peak_u_ratio_vs_cloak_size_{i}.png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        out_path,
        dpi=300,
    )
    plt.close(fig)